# HH GOA Voice RAG — Kaggle GPU build (v7)

Self-contained: backend code is embedded (split across cells). Builds the real `ai4bharat/MSMARCO-XI` index from the **train** split (hin+mar+eng).

**Runtime:** Accelerator = **GPU P100**. `HHGOA_SAMPLE_ROWS` = 60000 → ~2.7M chunks, ~60–90 min.


## 1) Check GPU


In [ ]:
import os, sys, time, shutil, subprocess, glob, zipfile, base64
t0 = time.perf_counter()
print('python:', sys.executable, flush=True)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout or 'NO NVIDIA GPU', flush=True)
B64_PARTS = []
print('cell1 OK', flush=True)


## 2) Install dependencies


In [ ]:
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'fastembed==0.5.1', 'onnxruntime-gpu==1.20.2', 'faiss-cpu',
    'datasets', 'pyarrow', 'fastapi', 'pydantic>=2.7', 'httpx', 'numpy'],
    capture_output=True, text=True)
print('pip rc:', r.returncode, flush=True)
print('pip err tail:', (r.stderr or '')[-1500:], flush=True)
assert r.returncode == 0, 'pip failed'


## 3) Embedded backend code (part 1/2)


In [ ]:
B64_PARTS.append("...")


In [ ]:
B64_PARTS.append("...")


In [ ]:
data = base64.b64decode(''.join(B64_PARTS))
open('/kaggle/working/backend.zip', 'wb').write(data)
with zipfile.ZipFile('/kaggle/working/backend.zip') as z:
    z.extractall('/kaggle/working')
os.chdir('/kaggle/working')
print('extracted backend:', sorted(os.listdir('backend')), flush=True)


## 4) Configure  
`HHGOA_SAMPLE_ROWS` = per-language variants (train split).


In [ ]:
os.environ['HHGOA_USE_REAL']          = '1'
os.environ['HHGOA_DATASET_SPLIT']     = 'train'
os.environ['HHGOA_DATASET_LANGUAGES'] = 'hin,mar'
os.environ['HHGOA_SAMPLE_ROWS']       = '60000'
os.environ['HHGOA_INDEX_ENGLISH']     = '1'
os.environ['HHGOA_EMBED_DEVICE']      = 'cuda'
os.environ['HHGOA_EMBED_BATCH']       = '256'
os.environ['HHGOA_IVF_NLIST']         = '1024'
os.environ['HHGOA_IVF_NPROBE']        = '32'
os.environ['HHGOA_IVF_TRAIN_SAMPLE']  = '100000'
os.environ['HHGOA_TOP_K']             = '6'
print('configured ok for 2.7M index', flush=True)


## 5) Build the index


In [ ]:
p = subprocess.Popen([sys.executable, '-u', '-m', 'backend.ingest', '--force'],
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, encoding='utf-8', errors='replace')
with open('/kaggle/working/ingest_progress.log', 'w', encoding='utf-8') as f:
    for line in p.stdout:
        print(line, end='', flush=True)
        f.write(line)
p.wait()
print('ingest exit code:', p.returncode, flush=True)
assert p.returncode == 0, 'ingest failed'


## 6) Latency benchmark (P50 / P70 / P100)


In [ ]:
subprocess.run([sys.executable, '-m', 'backend.benchmark', '--queries', '20',
                '--json', 'results/latency.json'], check=True)
print(f'Total elapsed: {round((time.perf_counter()-t0)/60, 1)} min', flush=True)


## 7) Download the built index


In [ ]:
if os.path.isdir('/kaggle/working/data/index'):
    shutil.make_archive('/kaggle/working/hhgoa_index', 'zip', '/kaggle/working/data/index')
    from IPython.display import FileLink
    display(FileLink('/kaggle/working/hhgoa_index.zip'))
    print('Download hhgoa_index.zip from right file panel -> /kaggle/working', flush=True)
else:
    print('index dir missing â€” build failed?', flush=True)
